# AI4MARS Terrain Segmentation: a Model Demo

This Colab walkthrough loads a small sample of the AI4MARS rover images, trains one compact U-Net, and shows predictions on held-out validation images. Data augmentation, experiment comparisons, and final test-set scoring are omitted to keep the notebook focused on the model demo.

**Quick-demo settings:** 128×128 images, a narrower U-Net, up to 1,200 records, and 5 epochs. Use a Colab GPU for a faster run. These settings are for a classroom showcase, not for reproducing the full project results.


## 1. Set up the runtime

The setup cell installs notebook dependencies, locates the repository (cloning it in Colab if needed), sets reproducibility options, and selects a GPU when available.

In [ ]:
%pip install -q datasets scikit-learn pillow matplotlib tqdm

import json
import random
import subprocess
import sys
from pathlib import Path
import numpy as np
import torch

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "train.py").is_file():
    PROJECT_DIR = Path("/content/ai4mars-segmentation")
    if not (PROJECT_DIR / "train.py").is_file():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/CatLover9463/ai4mars-segmentation", str(PROJECT_DIR)],
            check=True,
        )

# Edit experiment settings here.
SEED = 0
DATASET_ID = "hassanjbara/AI4MARS"
IMAGE_SIZE = (128, 128)
DEMO_MAX_RECORDS = 1200
NUM_CLASSES = 4
IGNORE_INDEX = 255
CLASS_NAMES = ["Soil", "Bedrock", "Sand", "Big rock"]
LABEL_ID_MAPPING = {0: 0, 1: 1, 2: 2, 3: 3}
BATCH_SIZE = 16
NUM_WORKERS = 2
BASE_CHANNELS = 16
LEARNING_RATE = 1e-3
NUM_EPOCHS = 5
OUTPUT_DIR = Path("outputs/ai4mars_walkthrough")

# Save outputs locally in the project workspace (including in Colab).
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repository: {PROJECT_DIR.resolve()}")
print(f"Runtime device: {device}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected; training will run on CPU and may be slow.")

## 2. Load and split the dataset

The dataset’s training partition is split into training and validation records with a fixed seed. Records with broken images, missing masks, or no usable class pixels are excluded. The official test partitions remain separate.

In [ ]:
# Dataset loading, cleaning, and splitting

from datasets import load_dataset, Image as HFImage
from sklearn.model_selection import train_test_split
from io import BytesIO
from PIL import Image


# Load the dataset.
train_data = load_dataset(DATASET_ID, split="train")
if len(train_data) > DEMO_MAX_RECORDS:
    train_data = train_data.shuffle(seed=SEED).select(range(DEMO_MAX_RECORDS))
print(f"Using {len(train_data):,} randomly selected records for this quick demo.")

# Inspect encoded image fields without decoding every image automatically.
raw_data = train_data.select_columns(
    ["image", "label_mask", "has_labels"]
).cast_column(
    "image", HFImage(decode=False)
).cast_column(
    "label_mask", HFImage(decode=False)
)

def decode_image_field(value):
    if value is None:
        return None

    if value.get("bytes") is not None:
        return Image.open(BytesIO(value["bytes"])).copy()

    if value.get("path"):
        return Image.open(value["path"]).copy()

    raise ValueError("Image has neither bytes nor a path.")

usable_indices = []
bad_image_indices = []
missing_mask_indices = []
empty_mask_indices = []

for idx in range(len(raw_data)):
    row = raw_data[idx]

    try:
        image = decode_image_field(row["image"])
        image.close()
    except Exception:
        bad_image_indices.append(idx)
        continue

    if not row["has_labels"] or row["label_mask"] is None:
        missing_mask_indices.append(idx)
        continue

    mask = decode_image_field(row["label_mask"]).convert("L")
    valid_pixels = np.isin(np.asarray(mask), list(LABEL_ID_MAPPING)).any()
    mask.close()

    if not valid_pixels:
        empty_mask_indices.append(idx)
        continue

    usable_indices.append(idx)

train_indices, val_indices = train_test_split(
    usable_indices,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

train_subset = train_data.select(train_indices)
val_subset = train_data.select(val_indices)

print("Total records:", len(train_data))
print("Usable records:", len(usable_indices))
print("Broken images:", len(bad_image_indices))
print("Missing masks:", len(missing_mask_indices))
print("Empty masks:", len(empty_mask_indices))
print("Training:", len(train_subset))
print("Validation:", len(val_subset))

assert set(train_indices).isdisjoint(val_indices)
assert len(train_indices) + len(val_indices) == len(usable_indices)

## 3. Build the data pipeline

Images and masks are resized to 128×128 for this quick demo. Image values become grayscale floats; masks keep integer class IDs, and unlabeled or unknown pixels use the ignore index.

In [ ]:
# PyTorch dataset and data loaders

import torch
from torch.utils.data import Dataset, DataLoader

class MarsTerrainDataset(Dataset):
    def __init__(self, hf_dataset, image_size=IMAGE_SIZE):
        self.data = hf_dataset
        self.image_size = image_size

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]

        image = row["image"].convert("L")
        mask = row["label_mask"].convert("L")

        image = image.resize(
            self.image_size,
            Image.Resampling.BILINEAR,
        )
        raw_mask = np.asarray(mask, dtype=np.uint8)
        mapped_mask = np.full(raw_mask.shape, IGNORE_INDEX, dtype=np.uint8)
        for source_id, class_id in LABEL_ID_MAPPING.items():
            mapped_mask[raw_mask == source_id] = class_id
        mask = Image.fromarray(mapped_mask, mode="L").resize(
            self.image_size, Image.Resampling.NEAREST
        )

        image_array = np.asarray(image, dtype=np.float32) / 255.0
        mask_array = np.asarray(mask, dtype=np.int64)

        x = torch.from_numpy(image_array.copy()).unsqueeze(0)
        y = torch.from_numpy(mask_array.copy())

        return x, y


train_dataset = MarsTerrainDataset(train_subset)
val_dataset = MarsTerrainDataset(val_subset)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

# Sanity check one batch.
images, masks = next(iter(train_loader))

print("Image batch:", images.shape)
print("Mask batch:", masks.shape)
print("Mask values:", torch.unique(masks).tolist())

assert images.shape[1:] == (1, *IMAGE_SIZE)
assert masks.shape[1:] == IMAGE_SIZE
assert torch.all(
    (masks == IGNORE_INDEX) |
    ((masks >= 0) & (masks < NUM_CLASSES))
)

print("Dataset pipeline checks passed.")

## 4. Inspect examples

Look at several image/mask pairs before training. This helps catch data or label issues early.

In [ ]:

# Visualize training images and terrain masks

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Visualization colors only; original label IDs remain unchanged.
terrain_cmap = ListedColormap([
    "#c49a6c",  # Soil
    "#64748b",  # Bedrock
    "#eab308",  # Sand
    "#dc2626",  # Big rock
])
terrain_cmap.set_bad(color="#d1d5db")  # Unlabeled pixels

NUM_SAMPLES = 3

fig, axes = plt.subplots(
    NUM_SAMPLES, 2,
    figsize=(12, 4 * NUM_SAMPLES),
    squeeze=False,
)

for i in range(NUM_SAMPLES):
    image_tensor, mask_tensor = train_dataset[i]

    image = image_tensor.squeeze(0).numpy()
    mask = mask_tensor.numpy()

    labels = torch.unique(mask_tensor).tolist()
    valid = mask != IGNORE_INDEX
    coverage = valid.mean() * 100

    print(
        f"Sample {i}: "
        f"image={tuple(image_tensor.shape)}, "
        f"mask={tuple(mask_tensor.shape)}, "
        f"labels={labels}, "
        f"labeled coverage={coverage:.1f}%"
    )

    # Original grayscale EDR image
    axes[i, 0].imshow(image, cmap="gray")
    axes[i, 0].set_title(f"EDR image {i}")
    axes[i, 0].axis("off")

    # Terrain mask, with unlabeled pixels shown in gray
    display_mask = np.ma.masked_equal(mask, IGNORE_INDEX)

    axes[i, 1].imshow(
        display_mask,
        cmap=terrain_cmap,
        vmin=0,
        vmax=3,
        interpolation="nearest",
    )
    axes[i, 1].set_title(
        f"Terrain mask {i} — {coverage:.1f}% labeled"
    )
    axes[i, 1].axis("off")

legend_items = [
    Patch(facecolor=terrain_cmap(i), label=name)
    for i, name in enumerate(CLASS_NAMES)
]
legend_items.append(
    Patch(facecolor="#d1d5db", label="Unlabeled")
)

fig.legend(
    handles=legend_items,
    loc="lower center",
    ncol=5,
    bbox_to_anchor=(0.5, 0.01),
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()


In [ ]:
import torch

# Inspect one real batch from the training data pipeline.
images, masks = next(iter(train_loader))

# Print the runtime, dataset sizes, tensor shapes, and label values.
print("Device:", device)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Image shape:", images.shape)
print("Mask shape:", masks.shape)
print("Image range:", images.min().item(), images.max().item())
print("Mask values:", torch.unique(masks).tolist())
print("NUM_CLASSES:", NUM_CLASSES)
print("IGNORE_INDEX:", IGNORE_INDEX)

# Check that images and masks have the expected batch/channel dimensions.
assert images.ndim == 4
assert images.shape[1] == 1
assert masks.ndim == 3
assert images.shape[0] == masks.shape[0]

# Check image values are finite and mask labels are valid or explicitly ignored.
assert torch.isfinite(images).all()
assert torch.all(
    (masks == IGNORE_INDEX) |
    ((masks >= 0) & (masks < NUM_CLASSES))
)

print("Batch checks passed.")

## 5. Evaluate the model

We use validation mean intersection over union (mIoU) as a compact summary of how well the model segments each terrain class. Pixel accuracy is shown as a secondary metric.


In [ ]:

# Reusable Segmentation Evaluation

import torch

@torch.no_grad()
def evaluate_segmentation(model, loader):
    model.eval()

    confusion_matrix = torch.zeros(
        NUM_CLASSES, NUM_CLASSES, dtype=torch.long
    )

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        predictions = logits.argmax(dim=1)

        # Ignore unlabeled and invalid target pixels
        valid = (
            (masks != IGNORE_INDEX)
            & (masks >= 0)
            & (masks < NUM_CLASSES)
        )

        targets = masks[valid].long()
        predictions = predictions[valid].long()

        if targets.numel() == 0:
            continue

        indices = targets * NUM_CLASSES + predictions

        confusion_matrix += torch.bincount(
            indices.cpu(),
            minlength=NUM_CLASSES**2,
        ).reshape(NUM_CLASSES, NUM_CLASSES)

    true_positives = confusion_matrix.diag().float()
    actual_counts = confusion_matrix.sum(dim=1).float()
    predicted_counts = confusion_matrix.sum(dim=0).float()

    union = actual_counts + predicted_counts - true_positives

    iou = torch.full((NUM_CLASSES,), float("nan"))
    present = union > 0
    iou[present] = true_positives[present] / union[present]

    total_pixels = confusion_matrix.sum().item()
    pixel_accuracy = (
        true_positives.sum().item() / total_pixels
        if total_pixels > 0 else float("nan")
    )
    mean_iou = (
        iou[present].mean().item()
        if present.any() else float("nan")
    )

    return {
        "pixel_accuracy": pixel_accuracy,
        "miou": mean_iou,
        "per_class_iou": iou,
        "confusion_matrix": confusion_matrix,
    }


# Convert a confusion matrix into pixel accuracy and per-class IoU.
def metrics_from_confusion_matrix(confusion_matrix):
    cm = confusion_matrix.to(torch.float64)

    true_positive = cm.diag()
    actual_count = cm.sum(dim=1)
    predicted_count = cm.sum(dim=0)

    union = actual_count + predicted_count - true_positive

    per_class_iou = torch.full_like(union, float("nan"))
    present = union > 0
    per_class_iou[present] = (
        true_positive[present] / union[present]
    )

    total_pixels = cm.sum()

    pixel_accuracy = (
        (true_positive.sum() / total_pixels).item()
        if total_pixels > 0
        else float("nan")
    )

    mean_iou = (
        per_class_iou[present].mean().item()
        if present.any()
        else float("nan")
    )

    return {
        "pixel_accuracy": pixel_accuracy,
        "per_class_iou": per_class_iou.cpu(),
        "miou": mean_iou,
        "valid_pixels": int(total_pixels.item()),
    }

print("Metric helper defined successfully.")



## 6. Build the U-Net

This U-Net uses encoder and decoder blocks with skip connections to combine terrain context with spatial detail. The demo uses 16 base channels to train faster; the full project configuration uses a wider model.


In [ ]:
# U-Net architecture: grayscale input and one output channel per terrain class

import torch
import torch.nn as nn


class DoubleConv(nn.Module):
    """Two convolution layers with BatchNorm and ReLU."""

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=3, padding=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=3, padding=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """Compact U-Net for semantic segmentation."""

    def __init__(self, in_channels=3, num_classes=4, base_channels=32):
        super().__init__()

        # Encoder: learn features while reducing spatial dimensions.
        self.enc1 = DoubleConv(in_channels, base_channels)
        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck: deepest feature representation.
        self.bottleneck = DoubleConv(
            base_channels * 4, base_channels * 8
        )

        # Decoder: upsample and combine encoder features.
        self.up3 = nn.ConvTranspose2d(
            base_channels * 8, base_channels * 4, 2, stride=2
        )
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(
            base_channels * 4, base_channels * 2, 2, stride=2
        )
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(
            base_channels * 2, base_channels, 2, stride=2
        )
        self.dec1 = DoubleConv(base_channels * 2, base_channels)

        # One output logit channel per terrain class.
        self.head = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        # Bottleneck
        b = self.bottleneck(self.pool(e3))

        # Decoder + skip connections
        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.head(d1)

In [ ]:
# Create the model used for the real-data checks and final training

model = UNet(
    in_channels=1,
    num_classes=NUM_CLASSES,
    base_channels=BASE_CHANNELS,
).to(device)

print("Model input channels:", model.enc1.block[0].in_channels)

## 7. Check the model before training

Run one forward pass on a real batch. This confirms the model returns four class-score maps aligned with the pixel labels.


In [ ]:
# Quick real-data forward pass: verify image, mask, and loss shapes

import torch.nn as nn

images, masks = next(iter(train_loader))
images = images.to(device)
masks = masks.to(device)

model.train()
logits = model(images)

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
loss = criterion(logits, masks)

print("Logits:", logits.shape)
print("Loss:", loss.item())

assert logits.shape == (
    images.shape[0],
    NUM_CLASSES,
    images.shape[2],
    images.shape[3],
)

assert torch.isfinite(loss), "Loss is not finite!"

print("Real-data forward pass successful.")

## 8. Train one compact model

The progress bars show batch-level progress for each training epoch. After each epoch, the notebook evaluates validation mIoU and keeps the best checkpoint in memory. Five epochs and the reduced model keep the demo manageable; scores should not be compared directly with the full project run.

In [ ]:
from tqdm.auto import tqdm
import copy
import time
import torch.nn as nn

# A fresh model for this demo run.
torch.manual_seed(SEED)
model = UNet(
    in_channels=1,
    num_classes=NUM_CLASSES,
    base_channels=BASE_CHANNELS,
).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = []
best_val_miou = -1.0
best_state = copy.deepcopy(model.state_dict())

for epoch in range(1, NUM_EPOCHS + 1):
    started = time.perf_counter()
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", leave=True)

    for batch_idx, (images, masks) in enumerate(progress, start=1):
        images = images.to(device, non_blocking=(device.type == "cuda"))
        masks = masks.to(device, non_blocking=(device.type == "cuda"))
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        progress.set_postfix(loss=f"{running_loss / batch_idx:.3f}")

    train_loss = running_loss / max(1, len(train_loader))
    val_metrics = evaluate_segmentation(model, val_loader)
    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_miou": val_metrics["miou"],
        "val_pixel_accuracy": val_metrics["pixel_accuracy"],
        "seconds": time.perf_counter() - started,
    }
    history.append(record)
    if val_metrics["miou"] > best_val_miou:
        best_val_miou = val_metrics["miou"]
        best_state = copy.deepcopy(model.state_dict())
    print(
        f"Epoch {epoch}/{NUM_EPOCHS}: train loss={train_loss:.4f}, "
        f"validation mIoU={val_metrics['miou']:.4f}, "
        f"pixel accuracy={val_metrics['pixel_accuracy']:.4f}, "
        f"time={record['seconds'] / 60:.1f} min"
    )

model.load_state_dict(best_state)
print(f"Best validation mIoU: {best_val_miou:.4f}")


## 9. Compare a success and a failure

The cell below scores every held-out validation example and displays the highest- and lowest-scoring cases. The displayed per-image mIoU makes both a success and a real limitation visible; these are validation examples, not test-set results.


In [ ]:
import matplotlib.pyplot as plt

PALETTE = np.array([[156,112,73], [126,132,137], [220,183,110], [200,70,55]], dtype=np.uint8)

def overlay(gray, labels):
    colors = PALETTE[np.clip(labels, 0, NUM_CLASSES - 1)]
    base = np.repeat(gray[..., None], 3, axis=2).astype(np.float32)
    result = base.copy()
    valid = labels != IGNORE_INDEX
    result[valid] = 0.52 * base[valid] + 0.48 * colors[valid]
    return result.clip(0, 255).astype(np.uint8)

def mean_iou_for_image(prediction, target):
    valid = target != IGNORE_INDEX
    class_ious = []
    for class_id in range(NUM_CLASSES):
        truth_class = (target == class_id) & valid
        predicted_class = (prediction == class_id) & valid
        union = np.logical_or(truth_class, predicted_class).sum()
        if union:
            intersection = np.logical_and(truth_class, predicted_class).sum()
            class_ious.append(intersection / union)
    return float(np.mean(class_ious)) if class_ious else float("nan")


# Score the held-out validation split only; the test split is never used here.
candidate_images = []
candidate_masks = []
for index in range(len(val_dataset)):
    image_tensor, mask_tensor = val_dataset[index]
    candidate_images.append(image_tensor)
    candidate_masks.append(mask_tensor.numpy())

scored_examples = []
model.eval()
with torch.inference_mode():
    for start in range(0, len(val_dataset), BATCH_SIZE):
        image_batch = torch.stack(candidate_images[start:start + BATCH_SIZE]).to(device)
        prediction_batch = model(image_batch).argmax(dim=1).cpu().numpy()
        for offset, prediction in enumerate(prediction_batch):
            index = start + offset
            score = mean_iou_for_image(prediction, candidate_masks[index])
            if np.isfinite(score):
                scored_examples.append((score, index, prediction))

if len(scored_examples) < 2:
    raise RuntimeError("Need at least two labeled validation examples to show a comparison.")

scored_examples.sort(key=lambda item: item[0])
failure = scored_examples[0]
success = scored_examples[-1]
print(f"Success example: validation index {success[1]}, per-image mIoU={success[0]:.3f}")
print(f"Failure example: validation index {failure[1]}, per-image mIoU={failure[0]:.3f}")

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for row, (label, example) in enumerate([
    ("Success", success),
    ("Failure", failure),
]):
    score, index, prediction = example
    image_tensor = candidate_images[index]
    gray = (image_tensor.squeeze(0).numpy() * 255).clip(0, 255).astype(np.uint8)
    truth = candidate_masks[index]

    axes[row, 0].imshow(gray, cmap="gray")
    axes[row, 0].set_title(f"{label}: input (val {index}, mIoU {score:.2f})")
    axes[row, 1].imshow(overlay(gray, truth))
    axes[row, 1].set_title("Reference labels")
    axes[row, 2].imshow(overlay(gray, prediction))
    axes[row, 2].set_title("U-Net prediction")
    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()
